# 🤟 Real-Time Sign Language Detector
### Architecture: MediaPipe Hand Landmarks → scikit-learn Classifier

**Why this approach?**
- ✅ No TensorFlow, no PyYAML build errors, no numpy conflicts
- ✅ Works perfectly on Python 3.10 + Windows + Conda
- ✅ MediaPipe extracts 21 hand landmarks (63 features) per frame
- ✅ A Random Forest / MLP on those landmarks is fast, accurate, and lightweight
- ✅ No GPU required

**Pipeline:**
```
Webcam → MediaPipe Hands → 63 landmark features → scikit-learn model → Predicted Sign
```

---
## Step 0 — Verify Environment

In [1]:
import sys
print(f"Python: {sys.version}")

import numpy as np
print(f"NumPy:  {np.__version__}")

import cv2
print(f"OpenCV: {cv2.__version__}")

import mediapipe as mp
print(f"MediaPipe: {mp.__version__}")

import sklearn
print(f"scikit-learn: {sklearn.__version__}")

print("\n✅ All dependencies verified successfully.")

Python: 3.10.20 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:42:35) [MSC v.1942 64 bit (AMD64)]
NumPy:  1.26.4
OpenCV: 4.11.0
MediaPipe: 0.10.9
scikit-learn: 1.4.2

✅ All dependencies verified successfully.


## Step 1 — Configuration
Edit the `SIGNS` list to match the hand signs you want to detect.

In [2]:
import os
import json

# ── USER CONFIGURATION ──────────────────────────────────────────
SIGNS =['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z','space']   # ← Add/remove signs here
SAMPLES_PER_SIGN = 200               # Images captured per sign
DATASET_PATH = r"archive (1)\ASL_Alphabet_Dataset\asl_alphabet_train"             # Folder to save raw images
LANDMARK_CSV = "landmarks.csv"       # Extracted feature file
MODEL_PATH = "sign_model.pkl"        # Trained model output
# ────────────────────────────────────────────────────────────────

# os.makedirs(DATASET_PATH, exist_ok=True)
# for sign in SIGNS:
#     os.makedirs(os.path.join(DATASET_PATH, sign), exist_ok=True)

# print(f"Signs to detect: {SIGNS}")
# print(f"Dataset folder ready: ./{DATASET_PATH}/")

## Step 2 — Dataset Capture

- A window will open showing your webcam feed.
- Press **`C`** to capture each image.
- Press **`ESC`** to exit early.
- Run this cell once per sign, changing the `label` argument.

In [4]:
def capture_data(label, num_samples=200, dataset_path="dataset"):
    """
    Capture webcam images for a given sign label.
    Press 'C' to capture, ESC to stop early.
    """
    save_path = os.path.join(dataset_path, label)
    os.makedirs(save_path, exist_ok=True)

    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        raise RuntimeError("Cannot open webcam. Check your camera index (try VideoCapture(1)).")

    count = 0
    print(f"[INFO] Collecting '{label}' — press C to capture, ESC to stop.")

    while count < num_samples:
        ret, frame = cap.read()
        if not ret:
            print("[WARN] Failed to read frame.")
            break

        display = frame.copy()
        # Status overlay
        cv2.rectangle(display, (0, 0), (frame.shape[1], 70), (0, 0, 0), -1)
        cv2.putText(display, f"Sign: {label}  [{count}/{num_samples}]",
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 100), 2)
        cv2.putText(display, "Press C to capture | ESC to stop",
                    (10, 58), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (200, 200, 200), 1)

        # ROI guide box (center of frame)
        h, w = frame.shape[:2]
        cx, cy = w // 2, h // 2
        cv2.rectangle(display, (cx - 150, cy - 150), (cx + 150, cy + 150), (0, 200, 255), 2)

        cv2.imshow("Dataset Capture", display)
        key = cv2.waitKey(1) & 0xFF

        if key == ord('c') or key == ord('C'):
            img_path = os.path.join(save_path, f"{label}_{count:04d}.jpg")
            cv2.imwrite(img_path, frame)
            count += 1
            print(f"  Saved {img_path}", end="\r")

        elif key == 27:  # ESC
            print(f"\n[INFO] Stopped early at {count} samples.")
            break

    cap.release()
    cv2.destroyAllWindows()
    print(f"\n✅ Done. {count} images saved for '{label}'.")

print("capture_data() function defined. Run the next cell to start capturing.")

capture_data() function defined. Run the next cell to start capturing.


In [9]:
# ── Run once per sign ───────────────────────────────────────────
# Change the label each time you run this cell.

capture_data("E", num_samples=SAMPLES_PER_SIGN)
# capture_data("B", num_samples=SAMPLES_PER_SIGN)
# capture_data("C", num_samples=SAMPLES_PER_SIGN)
# etc.

[INFO] Collecting 'E' — press C to capture, ESC to stop.
  Saved dataset\E\E_0199.jpg
✅ Done. 200 images saved for 'E'.


## Step 3 — Landmark Extraction

MediaPipe detects **21 hand keypoints** per hand.  
Each keypoint has `(x, y, z)` → **63 features total** per image.  
These are normalized to the hand bounding box, making them scale/position invariant.

In [4]:
import os
import csv
import mediapipe as mp
import cv2
import numpy as np
from tqdm import tqdm

mp_hands = mp.solutions.hands

def extract_landmarks_from_dataset(dataset_path, output_csv):
    """
    Walk all sign folders, run MediaPipe on each image,
    extract 63 normalized landmark features (translation + scale),
    save to CSV.
    """
    rows = []
    skipped = 0

    with mp_hands.Hands(
        static_image_mode=True,
        max_num_hands=1,
        min_detection_confidence=0.5
    ) as hands:

        sign_folders = sorted([
            d for d in os.listdir(dataset_path)
            if os.path.isdir(os.path.join(dataset_path, d))
        ])

        print(f"Found sign folders: {sign_folders}")

        for label in sign_folders:
            folder = os.path.join(dataset_path, label)
            images = [f for f in os.listdir(folder)
                      if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

            print(f"Processing '{label}': {len(images)} images")

            for img_file in tqdm(images, desc=f"  {label}", leave=False):
                img_path = os.path.join(folder, img_file)
                img = cv2.imread(img_path)

                if img is None:
                    skipped += 1
                    continue

                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                result = hands.process(img_rgb)

                if result.multi_hand_landmarks:
                    lm = result.multi_hand_landmarks[0].landmark

                    # ── Translation normalization (wrist-relative) ──
                    wrist = lm[0]

                    points = []
                    for point in lm:
                        points.append([
                            point.x - wrist.x,
                            point.y - wrist.y,
                            point.z - wrist.z
                        ])

                    points = np.array(points)  # shape (21, 3)

                    # ── 🔥 Scale normalization (IMPORTANT FIX) ──
                    scale = np.max(np.linalg.norm(points, axis=1))
                    if scale > 0:
                        points = points / scale

                    # Flatten to 63 features
                    features = points.flatten().tolist()

                    rows.append([label] + features)

                else:
                    skipped += 1

    # ── Write CSV ──
    header = ["label"] + [f"{axis}{i}" for i in range(21) for axis in ["x", "y", "z"]]

    with open(output_csv, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(header)
        writer.writerows(rows)

    print(f"\n[OK] Extraction complete!")
    print(f"   Rows saved : {len(rows)}")
    print(f"   Skipped    : {skipped}")
    print(f"   Saved to   : {output_csv}")

    return len(rows)


# ── Run extraction ──
extract_landmarks_from_dataset(DATASET_PATH, LANDMARK_CSV)

Found sign folders: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'space']
Processing 'A': 8458 images


Processing 'C': 8146 images


Processing 'D': 7629 images


Processing 'E': 7744 images


Processing 'F': 8031 images


...

Processing 'Y': 8178 images


Processing 'Z': 7410 images


Processing 'space': 7071 images



[OK] Extraction complete!
   Rows saved : 148531
   Skipped    : 64677
   Saved to   : landmarks.csv


148531

## Step 4 — Explore the Dataset

In [15]:
import csv
from collections import Counter

with open(LANDMARK_CSV, newline="") as f:
    reader = csv.reader(f)
    header = next(reader)
    data = list(reader)

labels = [row[0] for row in data]
label_counts = Counter(labels)

print(f"Total samples : {len(data)}")
print(f"Features      : {len(header) - 1}")
print(f"\nSamples per sign:")
for sign, count in sorted(label_counts.items()):
    bar = '█' * (count // 5)
    print(f"  {sign:4s} : {count:4d}  {bar}")

# Check for class imbalance
min_count = min(label_counts.values())
max_count = max(label_counts.values())
ratio = max_count / max(min_count, 1)
if ratio > 3:
    print(f"\n⚠️  Class imbalance detected (ratio {ratio:.1f}x). Consider collecting more data for under-represented signs.")
else:
    print(f"\n✅ Class balance looks good (ratio {ratio:.1f}x).")

Total samples : 137583
Features      : 63

Samples per sign:
  A    : 5265  ████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████

## Step 5 — Train the Classifier

We train two models and pick the best:
- **Random Forest** — fast, robust, great baseline
- **MLP (Neural Network)** — higher accuracy on more complex sign sets

No TensorFlow. No GPU. Just scikit-learn.

In [16]:
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

# ── Load data ────────────────────────────────────────────────────
with open(LANDMARK_CSV, newline="") as f:
    reader = csv.reader(f)
    header = next(reader)
    data = list(reader)

X = np.array([[float(v) for v in row[1:]] for row in data])
y_raw = [row[0] for row in data]

le = LabelEncoder()
y = le.fit_transform(y_raw)

print(f"Feature matrix : {X.shape}")
print(f"Classes        : {list(le.classes_)}")

# ── Train/test split ─────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")

# ── Model 1: Random Forest ────────────────────────────────────────
print("\n--- Training Random Forest ---")
rf_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_split=2,
        random_state=42,
        n_jobs=-1
    ))
])
rf_pipeline.fit(X_train, y_train)
rf_acc = rf_pipeline.score(X_test, y_test)
print(f"Random Forest Test Accuracy: {rf_acc*100:.2f}%")

# ── Model 2: MLP ──────────────────────────────────────────────────
print("\n--- Training MLP Classifier ---")
mlp_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", MLPClassifier(
        hidden_layer_sizes=(256, 128, 64),
        activation="relu",
        max_iter=500,
        random_state=42,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
        verbose=False
    ))
])
mlp_pipeline.fit(X_train, y_train)
mlp_acc = mlp_pipeline.score(X_test, y_test)
print(f"MLP Test Accuracy          : {mlp_acc*100:.2f}%")

# ── Pick best model ───────────────────────────────────────────────
if mlp_acc >= rf_acc:
    best_model = mlp_pipeline
    best_name = "MLP"
    best_acc = mlp_acc
else:
    best_model = rf_pipeline
    best_name = "Random Forest"
    best_acc = rf_acc

print(f"\n🏆 Best model: {best_name} ({best_acc*100:.2f}%)")

# ── Detailed report ───────────────────────────────────────────────
y_pred = best_model.predict(X_test)
print("\n" + "─"*50)
print(classification_report(y_test, y_pred, target_names=le.classes_))

# ── Save model + label encoder ────────────────────────────────────
joblib.dump({"model": best_model, "label_encoder": le}, MODEL_PATH)
print(f"✅ Model saved to: {MODEL_PATH}")

Feature matrix : (137583, 63)
Classes        : ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'space']
Train: 110066 | Test: 27517

--- Training Random Forest ---
Random Forest Test Accuracy: 99.19%

--- Training MLP Classifier ---
MLP Test Accuracy          : 99.46%

🏆 Best model: MLP (99.46%)

──────────────────────────────────────────────────
              precision    recall  f1-score   support

           A       1.00      1.00      1.00      1053
           B       1.00      1.00      1.00      1200
           C       0.99      0.99      0.99       972
           D       1.00      1.00      1.00      1194
           E       1.00      0.99      1.00      1082
           F       1.00      1.00      1.00      1494
           G       1.00      1.00      1.00      1205
           H       0.99      0.99      0.99      1176
           I       1.00      0.99      0.99      1143
           K       1.00      0.99    

## Step 6 — Confusion Matrix

In [13]:
import numpy as np

cm = confusion_matrix(y_test, y_pred)
classes = le.classes_

print("Confusion Matrix:")
print(f"{'':6s}" + "".join(f"{c:6s}" for c in classes))
print("      " + "─" * (6 * len(classes)))
for i, row in enumerate(cm):
    vals = "".join(
        f"{v:6d}" if i != j else f"\033[92m{v:6d}\033[0m"
        for j, v in enumerate(row)
    )
    print(f"{classes[i]:6s}| {vals}")

print("\n(Green = correct predictions)")

Confusion Matrix:
      A     B     C     D     E     F     G     H     I     J     K     L     M     N     O     P     Q     R     S     T     U     V     W     X     Y     space 
      ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
A     |   1047     0     4     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     1     1
B     |      0  1198     0     0     0     1     0     0     0     0     0     0     1     0     0     0     0     0     0     0     0     0     0     0     0     0
C     |      0     0   966     0     0     0     0     0     0     0     0     0     1     0     2     0     1     0     0     0     0     0     0     0     0     2
D     |      0     0     2  1191     0     0     0     0     0     0     0     0     0     0     0     0     0     0     0     1     0     0     0     0     0   

## Step 7 — Real-Time Prediction

- Loads the saved model.
- Opens webcam.
- Detects hand, extracts landmarks, predicts sign in real time.
- Press **`ESC`** to quit.

In [17]:
import cv2
import numpy as np
import joblib
import mediapipe as mp
from collections import deque

# ── Load model ───────────────────────────────────────────────────
bundle = joblib.load(MODEL_PATH)
model = bundle["model"]
le = bundle["label_encoder"]
print(f"✅ Model loaded. Classes: {list(le.classes_)}")

# ── MediaPipe setup ──────────────────────────────────────────────
mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils
mp_style = mp.solutions.drawing_styles

# Smoothing: use majority vote over last N frames
SMOOTHING_WINDOW = 7
prediction_buffer = deque(maxlen=SMOOTHING_WINDOW)

def get_landmark_features(hand_landmarks):
    lm = hand_landmarks.landmark
    wrist = lm[0]

    points = []
    for point in lm:
        points.append([
            point.x - wrist.x,
            point.y - wrist.y,
            point.z - wrist.z
        ])

    points = np.array(points)

    # SAME normalization as training
    scale = np.max(np.linalg.norm(points, axis=1))
    if scale > 0:
        points = points / scale

    return points.flatten().reshape(1, -1)

# ── Real-time loop ───────────────────────────────────────────────
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Cannot open webcam.")

cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

print("[INFO] Starting real-time detection. Press ESC to quit.")

with mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    model_complexity=1,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.6
) as hands:

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.flip(frame, 1)  # Mirror for natural feel
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = hands.process(rgb)

        label_text = "No hand detected"
        conf_text = ""
        color = (100, 100, 100)

        if result.multi_hand_landmarks:
            hand_lm = result.multi_hand_landmarks[0]

            # Draw landmarks
            mp_draw.draw_landmarks(
                frame, hand_lm,
                mp_hands.HAND_CONNECTIONS,
                mp_style.get_default_hand_landmarks_style(),
                mp_style.get_default_hand_connections_style()
            )

            # Predict
            features = get_landmark_features(hand_lm)
            pred_idx = model.predict(features)[0]
            probas = model.predict_proba(features)[0]
            confidence = probas[pred_idx]

            prediction_buffer.append(pred_idx)

            # Smoothed prediction (majority vote)
            from collections import Counter as _Counter
            smooth_pred = _Counter(prediction_buffer).most_common(1)[0][0]
            label_text = le.inverse_transform([smooth_pred])[0]
            conf_text = f"{confidence*100:.1f}%"
            color = (0, 220, 100) if confidence > 0.8 else (0, 165, 255)

        # ── HUD overlay ──────────────────────────────────────────
        h, w = frame.shape[:2]

        # Top bar
        cv2.rectangle(frame, (0, 0), (w, 80), (20, 20, 20), -1)
        cv2.putText(frame, "Sign Language Detector",
                    (15, 35), cv2.FONT_HERSHEY_DUPLEX, 0.9, (200, 200, 200), 1)
        cv2.putText(frame, "ESC = Quit",
                    (w - 150, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (130, 130, 130), 1)

        # Prediction box (bottom-left)
        cv2.rectangle(frame, (0, h - 100), (380, h), (20, 20, 20), -1)
        cv2.putText(frame, f"Sign: {label_text}",
                    (15, h - 60), cv2.FONT_HERSHEY_DUPLEX, 1.2, color, 2)
        if conf_text:
            cv2.putText(frame, f"Conf: {conf_text}",
                        (15, h - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.75, color, 2)

        cv2.imshow("Sign Language Detector", frame)

        if cv2.waitKey(1) & 0xFF == 27:
            break

cap.release()
cv2.destroyAllWindows()
print("[INFO] Session ended.")

✅ Model loaded. Classes: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'space']
[INFO] Starting real-time detection. Press ESC to quit.
[INFO] Session ended.


## Step 8 — Evaluate on Saved Images (Optional)

Run inference on a directory of test images without opening the webcam.

In [8]:
import cv2
import numpy as np
import joblib
import mediapipe as mp

# ── Load model ───────────────────────────────────────────────────
bundle = joblib.load(MODEL_PATH)
model = bundle["model"]
le = bundle["label_encoder"]
print(f"✅ Model loaded. Classes: {list(le.classes_)}")

# ── MediaPipe setup ──────────────────────────────────────────────
mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils
mp_style = mp.solutions.drawing_styles

def get_landmark_features(hand_landmarks):
    lm = hand_landmarks.landmark
    wrist = lm[0]
    points = []
    for point in lm:
        points.append([
            point.x - wrist.x,
            point.y - wrist.y,
            point.z - wrist.z
        ])
    points = np.array(points)
    scale = np.max(np.linalg.norm(points, axis=1))
    if scale > 0:
        points = points / scale
    return points.flatten().reshape(1, -1)

# ── Single image prediction ──────────────────────────────────────
IMAGE_PATH = r"C:\Users\Administrator\Pictures\Camera Roll\e.jpg"

# Load image
img_original = cv2.imread(IMAGE_PATH)
if img_original is None:
    raise FileNotFoundError(f"Could not load image: {IMAGE_PATH}")

# Pad bottom (helps when wrist is cut off) then resize
img_padded = cv2.copyMakeBorder(
    img_original,
    top=0, bottom=150, left=0, right=0,
    borderType=cv2.BORDER_REPLICATE
)
img = cv2.resize(img_padded, (640, 800))
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# ── Run detection ────────────────────────────────────────────────
with mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=0.1
) as hands:

    result = hands.process(img_rgb)

    if not result.multi_hand_landmarks:
        print("❌ No hand detected.")
        print("   Tip: This image may not be compatible with MediaPipe.")
        print("   Best results come from webcam captures matching your training data.")
        cv2.imshow("Image (no detection)", img)
        cv2.waitKey(0)
        cv2.destroyAllWindows()

    else:
        hand_lm = result.multi_hand_landmarks[0]

        # Draw landmarks
        mp_draw.draw_landmarks(
            img, hand_lm,
            mp_hands.HAND_CONNECTIONS,
            mp_style.get_default_hand_landmarks_style(),
            mp_style.get_default_hand_connections_style()
        )

        # Predict
        features = get_landmark_features(hand_lm)
        pred_idx = model.predict(features)[0]
        probas = model.predict_proba(features)[0]
        confidence = probas[pred_idx]
        label = le.inverse_transform([pred_idx])[0]

        print(f"✅ Predicted Sign : {label}")
        print(f"   Confidence     : {confidence*100:.1f}%")

        # Draw HUD on image
        h, w = img.shape[:2]
        color = (0, 220, 100) if confidence > 0.8 else (0, 165, 255)
        cv2.rectangle(img, (0, 0), (w, 70), (20, 20, 20), -1)
        cv2.putText(img, f"Sign: {label}  ({confidence*100:.1f}%)",
                    (15, 45), cv2.FONT_HERSHEY_DUPLEX, 1.2, color, 2)

        cv2.imshow("Prediction Result", img)
        cv2.waitKey(0)
        cv2.destroyAllWindows()


✅ Model loaded. Classes: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'space']
✅ Predicted Sign : B
   Confidence     : 99.9%


In [14]:
import csv

input_csv = LANDMARK_CSV       # your existing landmarks.csv
output_csv = "landmarks_no_z.csv"

with open(input_csv, newline="") as fin, open(output_csv, "w", newline="") as fout:
    reader = csv.reader(fin)
    writer = csv.writer(fout)
    
    header = next(reader)
    writer.writerow(header)
    
    removed = 0
    kept = 0
    for row in reader:
        if row[0] == "":
            removed += 1
        else:
            writer.writerow(row)
            kept += 1

print(f"✅ Done.")
print(f"   Kept   : {kept} rows")
print(f"   Removed: {removed} Z rows")
print(f"   Saved to: {output_csv}")

✅ Done.
   Kept   : 137583 rows
   Removed: 5558 Z rows
   Saved to: landmarks_no_z.csv


In [ ]:
import csv
import os
import numpy as np
import mediapipe as mp
import cv2
from tqdm import tqdm

mp_hands = mp.solutions.hands

def extract_single_label(label, dataset_path, output_csv):
    rows = []
    skipped = 0
    folder = os.path.join(dataset_path, label)
    images = [f for f in os.listdir(folder)
              if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    print(f"Processing '{label}': {len(images)} images")

    with mp_hands.Hands(
        static_image_mode=True,
        max_num_hands=1,
        min_detection_confidence=0.5
    ) as hands:
        for img_file in tqdm(images, desc=f"  {label}"):
            img_path = os.path.join(folder, img_file)
            img = cv2.imread(img_path)
            if img is None:
                skipped += 1
                continue

            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            result = hands.process(img_rgb)

            if result.multi_hand_landmarks:
                lm = result.multi_hand_landmarks[0].landmark
                wrist = lm[0]
                features = []
                for point in lm:
                    features.extend([
                        point.x - wrist.x,
                        point.y - wrist.y,
                        point.z - wrist.z
                    ])
                rows.append([label] + features)
            else:
                skipped += 1

    # ── Append to existing CSV (no header) ──────────────────────
    with open(output_csv, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerows(rows)

    print(f"✅ Done. Added {len(rows)} Z rows, skipped {skipped}")


# ── Run it ───────────────────────────────────────────────────────
extract_single_label("Z", DATASET_PATH, LANDMARK_CSV)

In [6]:
cv2.imshow("Debug", img)
cv2.waitKey(0)
cv2.destroyAllWindows()